## Configure Source Paths


In [0]:

CATALOG = "capretailnew"
SCHEMA = "retail"
VOLUME = "ext_retail_vol"

ROOT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

SALES_PATH = ROOT_PATH + "/sales_order_lines"
INVENTORY_PATH = ROOT_PATH + "/inventory_movements"
RETURNS_PATH = ROOT_PATH + "/sales_returns"
display(dbutils.fs.ls(ROOT_PATH))

path,name,size,modificationTime
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/checkpoints/,checkpoints/,0,1793805117000
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/inventory_movements/,inventory_movements/,0,1793766126000
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/logs/,logs/,0,1793767245000
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/mlflow_tmp/,mlflow_tmp/,0,1793856649000
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/quarantine/,quarantine/,0,1793767245000
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/reference/,reference/,0,1793766191000
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/references1/,references1/,0,1793799841000
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/sales_order_lines/,sales_order_lines/,0,1793766059000
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/sales_returns/,sales_returns/,0,1793766154000
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/schema_tracking/,schema_tracking/,0,1793805118000


## Check Source Folders

In [0]:
print("Sales Files")
display(dbutils.fs.ls(SALES_PATH))

print("Inventory Files")
display(dbutils.fs.ls(INVENTORY_PATH))

print("Returns Files")
display(dbutils.fs.ls(RETURNS_PATH))

Sales Files


path,name,size,modificationTime
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/sales_order_lines/sales_order_lines_batch_001.csv,sales_order_lines_batch_001.csv,152400,1793792273000
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/sales_order_lines/sales_order_lines_batch_002.csv,sales_order_lines_batch_002.csv,152556,1793792288000
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/sales_order_lines/sales_order_lines_batch_003.csv,sales_order_lines_batch_003.csv,152648,1793792301000
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/sales_order_lines/sales_order_lines_batch_004.csv,sales_order_lines_batch_004.csv,152733,1793792320000
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/sales_order_lines/sales_order_lines_batch_005.csv,sales_order_lines_batch_005.csv,152539,1793792339000
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/sales_order_lines/sales_order_lines_batch_006.csv,sales_order_lines_batch_006.csv,152399,1793792352000
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/sales_order_lines/sales_order_lines_batch_007.csv,sales_order_lines_batch_007.csv,152553,1793792371000
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/sales_order_lines/sales_order_lines_batch_008.csv,sales_order_lines_batch_008.csv,152635,1793792388000
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/sales_order_lines/sales_order_lines_batch_009.csv,sales_order_lines_batch_009.csv,152751,1793792397000
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/sales_order_lines/sales_order_lines_batch_010.csv,sales_order_lines_batch_010.csv,151538,1793792409000


Inventory Files


path,name,size,modificationTime
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/inventory_movements/inventory_movements.csv,inventory_movements.csv,88284,1793792591000


Returns Files


path,name,size,modificationTime
dbfs:/Volumes/capretailnew/retail/ext_retail_vol/sales_returns/sales_returns.csv,sales_returns.csv,35840,1793792481000


## Create Required folders

In [0]:
folders = [
    "checkpoints/sales",
    "checkpoints/inventory",
    "checkpoints/returns",
    "schema_tracking/sales",
    "schema_tracking/inventory",
    "schema_tracking/returns",
    "logs",
    "quarantine"
]

for folder in folders:
    dbutils.fs.mkdirs(ROOT_PATH + "/" + folder)

print("Required folders created successfully")

Required folders created successfully


### Import Required Libraries

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    when,
    trim,
    lit,
    sum,
    min,
    max,
    avg
)

# Read Sales CSV Files

sales_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(SALES_PATH)
)

print("Sales Record Count:", sales_df.count())

display(sales_df)

Sales Record Count: 11412


order_line_id,order_id,order_ts,store_id,product_id,customer_id,quantity,unit_price,discount_pct,sales_channel,order_status,tax_amount
ORDER_L040009137,ORDER040009137,2026-08-14T12:13:00Z,STORE0400010,PRODUC0400006,CUSTOME040009137,2,15180.00,8.38,APP,PAID,15232.00
ORDER_L040009138,ORDER040009138,2026-08-14T12:20:00Z,STORE0400118,PRODUC0400035,CUSTOME040009138,6,15217.00,15.54,STORE,SHIPPED,15269.00
ORDER_L040009139,ORDER040009139,2026-08-14T12:27:00Z,STORE0400024,PRODUC0400133,CUSTOME040009139,1,15254.00,19.98,STORE,RETURNED,15306.00
ORDER_L040009140,ORDER040009140,2026-08-14T12:34:00Z,STORE0400101,PRODUC0400104,CUSTOME040009140,5,15291.00,26.08,STORE,PAID,15343.00
ORDER_L040009141,ORDER040009141,2026-08-14T12:41:00Z,STORE0400012,PRODUC0400114,CUSTOME040009141,1,15328.00,25.70,APP,RETURNED,15380.00
ORDER_L040009142,ORDER040009142,2026-08-14T12:48:00Z,STORE0400049,PRODUC0400115,CUSTOME040009142,2,15365.00,8.17,WEB,PAID,15417.00
ORDER_L040009143,ORDER040009143,2026-08-14T12:55:00Z,STORE0400050,PRODUC0400020,CUSTOME040009143,4,15402.00,4.06,APP,RETURNED,15454.00
ORDER_L040009144,ORDER040009144,2026-08-14T13:02:00Z,STORE0400143,PRODUC0400054,CUSTOME040009144,7,15439.00,21.16,APP,PAID,15491.00
ORDER_L040009145,ORDER040009145,2026-08-14T13:09:00Z,STORE0400003,PRODUC0400087,CUSTOME040009145,1,15476.00,17.70,APP,RETURNED,15528.00
ORDER_L040009146,ORDER040009146,2026-08-14T13:16:00Z,STORE0400008,PRODUC0400138,CUSTOME040009146,3,15513.00,11.31,APP,RETURNED,15565.00


In [0]:
# DBTITLE 1,6. Check Sales Schema

sales_df.printSchema()

root
 |-- order_line_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_pct: string (nullable = true)
 |-- sales_channel: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- tax_amount: string (nullable = true)



In [0]:
#  Check Sales Columns

print("Sales Columns:")
print(sales_df.columns)

Sales Columns:
['order_line_id', 'order_id', 'order_ts', 'store_id', 'product_id', 'customer_id', 'quantity', 'unit_price', 'discount_pct', 'sales_channel', 'order_status', 'tax_amount']


In [0]:
#  Check Sales Missing Values

sales_missing = sales_df.select([
    count(
        when(
            col(c).isNull() | (trim(col(c)) == ""),
            c
        )
    ).alias(c)
    for c in sales_df.columns
])

display(sales_missing)

order_line_id,order_id,order_ts,store_id,product_id,customer_id,quantity,unit_price,discount_pct,sales_channel,order_status,tax_amount
0,2,0,0,0,1,3,2,2,3,1,0


In [0]:
# Check Duplicate Sales Records
print("Total Sales Records:", sales_df.count())

print(
    "Distinct Order Line IDs:",
    sales_df.select("order_line_id").distinct().count()
)

duplicate_sales = (
    sales_df.groupBy("order_line_id")
    .count()
    .filter(col("count") > 1)
)

print("Duplicate Order Line IDs:")
display(duplicate_sales)


Total Sales Records: 11412
Distinct Order Line IDs: 11373
Duplicate Order Line IDs:


order_line_id,count
ORDER_L040010250,2
ORDER_L040009477,2
ORDER_L040009744,2
ORDER_L040009854,2
ORDER_L040002685,2
ORDER_L040003139,2
ORDER_L040008840,2
ORDER_L040003337,2
ORDER_L040008043,2
ORDER_L040002956,2


In [0]:
# DBTITLE 1,10. Check Sales Data Distribution
display(
    sales_df.groupBy("order_status")
    .count()
)

order_status,count
PAID,3821
RETURNED,3819
SHIPPED,3771
null,1


In [0]:
# DBTITLE 1,11. Check Sales Numeric Columns
sales_numeric_df = (
    sales_df
    .withColumn("quantity_num", col("quantity").cast("double"))
    .withColumn("unit_price_num", col("unit_price").cast("double"))
    .withColumn("discount_pct_num", col("discount_pct").cast("double"))
)

display(
    sales_numeric_df.select(
        "quantity_num",
        "unit_price_num",
        "discount_pct_num"
    ).summary()
)

summary,quantity_num,unit_price_num,discount_pct_num
count,11409,11410,11410
mean,3.2150933473573495,11556.06308150745,14.97422787028923
stddev,42.08019517740012,45714.648982381885,8.672843447637115
min,-999.0,26.0,0.01
25%,3.0,4693.0,7.41
50%,5.0,9482.0,14.97
75%,7.0,14273.0,22.49
max,9.0,999999.99,30.0


In [0]:
#  Read Inventory Data
inventory_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(INVENTORY_PATH)
)

print("Inventory Record Count:", inventory_df.count())

display(inventory_df)

Inventory Record Count: 634


movement_id,product_id,store_id,movement_ts,movement_type,quantity_change,reference_id,unit_cost,warehouse_id,recorded_ts
MOVEMEN040000001,PRODUC0400127,STORE0400037,2026-07-01T02:26:00Z,SALE,1,REFEREN040000001,148.00,WAREHOU040000037,2026-07-01T02:56:00Z
MOVEMEN040000002,PRODUC0400015,STORE0400053,2026-07-01T02:33:00Z,SALE,-1,REFEREN040000002,185.00,WAREHOU040000053,2026-07-01T03:03:00Z
MOVEMEN040000003,PRODUC0400044,STORE0400050,2026-07-01T02:40:00Z,SALE,-8,REFEREN040000003,222.00,WAREHOU040000050,2026-07-01T03:10:00Z
MOVEMEN040000004,PRODUC0400061,STORE0400119,2026-07-01T02:47:00Z,TRANSFER,3,REFEREN040000004,259.00,WAREHOU040000119,2026-07-01T03:17:00Z
MOVEMEN040000005,PRODUC0400049,STORE0400004,2026-07-01T02:54:00Z,RECEIPT,-4,REFEREN040000005,296.00,WAREHOU040000004,2026-07-01T03:24:00Z
MOVEMEN040000006,PRODUC0400070,STORE0400085,2026-07-01T03:01:00Z,TRANSFER,-4,REFEREN040000006,333.00,WAREHOU040000085,2026-07-01T03:31:00Z
MOVEMEN040000007,PRODUC0400102,STORE0400050,2026-07-01T03:08:00Z,ADJUSTMENT,-8,REFEREN040000007,370.00,WAREHOU040000050,2026-07-01T03:38:00Z
MOVEMEN040000008,PRODUC0400072,STORE0400142,2026-07-01T03:15:00Z,SALE,3,REFEREN040000008,407.00,WAREHOU040000142,2026-07-01T03:45:00Z
MOVEMEN040000009,PRODUC0400136,STORE0400017,2026-07-01T03:22:00Z,SALE,1,REFEREN040000009,444.00,WAREHOU040000017,2026-07-01T03:52:00Z
MOVEMEN040000010,PRODUC0400079,STORE0400043,2026-07-01T03:29:00Z,SALE,7,REFEREN040000010,481.00,WAREHOU040000043,2026-07-01T03:59:00Z


In [0]:
# DBTITLE 1,13. Check Inventory Schema
inventory_df.printSchema()

root
 |-- movement_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- movement_ts: string (nullable = true)
 |-- movement_type: string (nullable = true)
 |-- quantity_change: string (nullable = true)
 |-- reference_id: string (nullable = true)
 |-- unit_cost: string (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- recorded_ts: string (nullable = true)



In [0]:
# DBTITLE 1,14. Check Inventory Missing Values
inventory_missing = inventory_df.select([
    count(
        when(
            col(c).isNull() | (trim(col(c)) == ""),
            c
        )
    ).alias(c)
    for c in inventory_df.columns
])

display(inventory_missing)

movement_id,product_id,store_id,movement_ts,movement_type,quantity_change,reference_id,unit_cost,warehouse_id,recorded_ts
0,0,0,0,0,0,0,0,0,0


In [0]:
# DBTITLE 1,15. Check Inventory Duplicate Records
total_count=inventory_df.count()
print("Total Inventory Records:", total_count)
distinct_count=inventory_df.select("movement_id").distinct().count()
print(
    "Distinct Movement IDs:",
    distinct_count
)
duplicate_count=total_count-distinct_count
print("duplicate count",duplicate_count)
duplicate_inventory = (
    inventory_df.groupBy("movement_id")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_inventory)

Total Inventory Records: 634
Distinct Movement IDs: 634
duplicate count 0


movement_id,count


In [0]:
# DBTITLE 1,16. Check Inventory Movement Types
display(
    inventory_df.groupBy("movement_type")
    .count()
)

movement_type,count
ADJUSTMENT,151
SALE,165
RECEIPT,136
TRANSFER,182


In [0]:
# DBTITLE 1,17. Read Returns Data

returns_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(RETURNS_PATH)
)

print("Returns Record Count:", returns_df.count())

display(returns_df)

Returns Record Count: 316


return_id,order_line_id,return_ts,return_quantity,refund_amount,return_reason,inspection_status,store_id,product_id,approved_flag
RETURN040000001,ORDER_L040000030,2026-07-01T02:21:00Z,9,109.00,WRONG_ITEM,INACTIVE,STORE0400053,PRODUC0400103,Y
RETURN040000002,ORDER_L040000059,2026-07-01T02:28:00Z,7,146.00,WRONG_ITEM,PENDING,STORE0400005,PRODUC0400099,Y
RETURN040000003,ORDER_L040000088,2026-07-01T02:35:00Z,3,183.00,DAMAGED,PENDING,STORE0400045,PRODUC0400082,Y
RETURN040000004,ORDER_L040000117,2026-07-01T02:42:00Z,3,220.00,CUSTOMER_REQUEST,ACTIVE,STORE0400074,PRODUC0400044,Y
RETURN040000005,ORDER_L040000146,2026-07-01T02:49:00Z,6,257.00,DAMAGED,INACTIVE,STORE0400068,PRODUC0400103,N
RETURN040000006,ORDER_L040000175,2026-07-01T02:56:00Z,6,294.00,DAMAGED,PENDING,STORE0400112,PRODUC0400134,Y
RETURN040000007,ORDER_L040000204,2026-07-01T03:03:00Z,5,331.00,DAMAGED,INACTIVE,STORE0400072,PRODUC0400003,Y
RETURN040000008,ORDER_L040000233,2026-07-01T03:10:00Z,1,368.00,CUSTOMER_REQUEST,ACTIVE,STORE0400073,PRODUC0400105,N
RETURN040000009,ORDER_L040000262,2026-07-01T03:17:00Z,8,405.00,WRONG_ITEM,ACTIVE,STORE0400058,PRODUC0400006,N
RETURN040000010,ORDER_L040000291,2026-07-01T03:24:00Z,9,442.00,CUSTOMER_REQUEST,INACTIVE,STORE0400096,PRODUC0400084,Y


In [0]:
# DBTITLE 1,18. Check Returns Schema

returns_df.printSchema()

root
 |-- return_id: string (nullable = true)
 |-- order_line_id: string (nullable = true)
 |-- return_ts: string (nullable = true)
 |-- return_quantity: string (nullable = true)
 |-- refund_amount: string (nullable = true)
 |-- return_reason: string (nullable = true)
 |-- inspection_status: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- approved_flag: string (nullable = true)



In [0]:
# DBTITLE 1,19. Check Returns Missing Values

returns_missing = returns_df.select([
    count(
        when(
            col(c).isNull() | (trim(col(c)) == ""),
            c
        )
    ).alias(c)
    for c in returns_df.columns
])

display(returns_missing)

return_id,order_line_id,return_ts,return_quantity,refund_amount,return_reason,inspection_status,store_id,product_id,approved_flag
0,0,0,0,0,0,0,0,0,0


In [0]:
# COMMAND ----------

# DBTITLE 1,20. Check Returns Duplicate Records

print("Total Returns Records:", returns_df.count())

print(
    "Distinct Return IDs:",
    returns_df.select("return_id").distinct().count()
)

duplicate_returns = (
    returns_df.groupBy("return_id")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_returns)




Total Returns Records: 316
Distinct Return IDs: 316


return_id,count


In [0]:
from pyspark.sql.functions import col

def find_outliers(sales_numeric_df, column_name):

    # Calculate Q1 and Q3
    q1, q3 = sales_numeric_df.approxQuantile(
        column_name, [0.25, 0.75], 0.01
    )

    # Calculate IQR
    iqr = q3 - q1

    # Calculate boundaries
    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    # Identify outliers
    outliers_df = sales_numeric_df.filter(
        (col(column_name) < lower_bound) |
        (col(column_name) > upper_bound)
    )

    print("Column:", column_name)
    print("Q1:", q1)
    print("Q3:", q3)
    print("IQR:", iqr)
    print("Lower Bound:", lower_bound)
    print("Upper Bound:", upper_bound)
    print("Outlier Count:", outliers_df.count())

    display(outliers_df)

    return outliers_df

In [0]:
numeric_columns = [
    "quantity_num",
    "unit_price_num",
    "discount_pct_num"
]

for column_name in numeric_columns:
    find_outliers(sales_numeric_df, column_name)

Column: quantity_num
Q1: 3.0
Q3: 7.0
IQR: 4.0
Lower Bound: -3.0
Upper Bound: 13.0
Outlier Count: 20


order_line_id,order_id,order_ts,store_id,product_id,customer_id,quantity,unit_price,discount_pct,sales_channel,order_status,tax_amount,quantity_num,unit_price_num,discount_pct_num
ORDER_L040009458,ORDER040009458,2026-08-16T01:40:00Z,STORE0400107,PRODUC0400115,CUSTOME040009458,-999.00,8057.00,14.46,WEB,RETURNED,8109.00,-999.0,8057.0,14.46
ORDER_L040010274,ORDER040010274,2026-08-20T00:52:00Z,STORE0400081,PRODUC0400037,CUSTOME040010274,-999.00,249.00,14.04,APP,RETURNED,301.00,-999.0,249.0,14.04
ORDER_L040003695,ORDER040003695,2026-07-19T01:19:00Z,STORE0400059,PRODUC0400099,CUSTOME040003695,-999.00,3826.00,0.98,APP,SHIPPED,3878.00,-999.0,3826.0,0.98
ORDER_L040004173,ORDER040004173,2026-07-21T09:05:00Z,STORE0400135,PRODUC0400087,CUSTOME040004173,-999.00,2512.00,5.60,WEB,PAID,2564.00,-999.0,2512.0,5.6
ORDER_L040002574,ORDER040002574,2026-07-13T14:32:00Z,STORE0400063,PRODUC0400039,CUSTOME040002574,-999.00,349.00,0.47,APP,SHIPPED,401.00,-999.0,349.0,0.47
ORDER_L040003356,ORDER040003356,2026-07-17T09:46:00Z,STORE0400079,PRODUC0400015,CUSTOME040003356,-999.00,10283.00,7.91,STORE,PAID,10335.00,-999.0,10283.0,7.91
ORDER_L040001261,ORDER040001261,2026-07-07T05:21:00Z,STORE0400123,PRODUC0400006,CUSTOME040001261,-999.00,8768.00,25.05,WEB,SHIPPED,8820.00,-999.0,8768.0,25.05
ORDER_L040006917,ORDER040006917,2026-08-03T17:13:00Z,STORE0400103,PRODUC0400025,CUSTOME040006917,-999.00,9040.00,11.66,WEB,PAID,9092.00,-999.0,9040.0,11.66
ORDER_L040007033,ORDER040007033,2026-08-04T06:45:00Z,STORE0400096,PRODUC0400131,CUSTOME040007033,-999.00,13332.00,6.80,WEB,SHIPPED,13384.00,-999.0,13332.0,6.8
ORDER_L040004750,ORDER040004750,2026-07-24T04:24:00Z,STORE0400119,PRODUC0400081,CUSTOME040004750,-999.00,4861.00,15.48,APP,RETURNED,4913.00,-999.0,4861.0,15.48


Column: unit_price_num
Q1: 4676.0
Q3: 14106.0
IQR: 9430.0
Lower Bound: -9469.0
Upper Bound: 28251.0
Outlier Count: 24


order_line_id,order_id,order_ts,store_id,product_id,customer_id,quantity,unit_price,discount_pct,sales_channel,order_status,tax_amount,quantity_num,unit_price_num,discount_pct_num
ORDER_L040009270,ORDER040009270,2026-08-15T03:44:00Z,STORE0400014,PRODUC0400030,CUSTOME040009270,4,999999.99,19.28,APP,SHIPPED,1153.00,4.0,999999.99,19.28
ORDER_L040009670,ORDER040009670,2026-08-17T02:24:00Z,STORE0400070,PRODUC0400027,CUSTOME040009670,2,999999.99,3.49,APP,RETURNED,15953.00,2.0,999999.99,3.49
ORDER_L040009930,ORDER040009930,2026-08-18T08:44:00Z,STORE0400101,PRODUC0400043,CUSTOME040009930,2,999999.99,5.88,APP,RETURNED,6573.00,2.0,999999.99,5.88
ORDER_L040004500,ORDER040004500,2026-07-22T23:14:00Z,STORE0400072,PRODUC0400048,CUSTOME040004500,6,999999.99,21.72,WEB,RETURNED,14663.00,6.0,999999.99,21.72
ORDER_L040002418,ORDER040002418,2026-07-12T20:20:00Z,STORE0400129,PRODUC0400138,CUSTOME040002418,1,999999.99,11.95,APP,SHIPPED,13629.00,1.0,999999.99,11.95
ORDER_L040002695,ORDER040002695,2026-07-14T04:39:00Z,STORE0400035,PRODUC0400079,CUSTOME040002695,8,999999.99,15.75,WEB,SHIPPED,4878.00,8.0,999999.99,15.75
ORDER_L040008297,ORDER040008297,2026-08-10T10:13:00Z,STORE0400075,PRODUC0400007,CUSTOME040008297,8,999999.99,0.01,STORE,SHIPPED,3152.00,8.0,999999.99,0.01
ORDER_L040008333,ORDER040008333,2026-08-10T14:25:00Z,STORE0400011,PRODUC0400075,CUSTOME040008333,6,999999.99,5.52,WEB,RETURNED,4484.00,6.0,999999.99,5.52
ORDER_L040009032,ORDER040009032,2026-08-13T23:58:00Z,STORE0400021,PRODUC0400090,CUSTOME040009032,4,999999.99,12.74,APP,RETURNED,11347.00,4.0,999999.99,12.74
ORDER_L040002222,ORDER040002222,2026-07-11T21:28:00Z,STORE0400123,PRODUC0400065,CUSTOME040002222,4,999999.99,6.53,APP,RETURNED,6377.00,4.0,999999.99,6.53


Column: discount_pct_num
Q1: 7.45
Q3: 22.22
IQR: 14.77
Lower Bound: -14.705000000000002
Upper Bound: 44.375
Outlier Count: 0


order_line_id,order_id,order_ts,store_id,product_id,customer_id,quantity,unit_price,discount_pct,sales_channel,order_status,tax_amount,quantity_num,unit_price_num,discount_pct_num


In [0]:
from pyspark.sql.functions import col
invent_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(INVENTORY_PATH)
)
def find_outliers(invent_df, column_name):

    # Calculate Q1 and Q3
    q1, q3 = invent_df.approxQuantile(
        column_name, [0.25, 0.75], 0.01
    )

    # Calculate IQR
    iqr = q3 - q1

    # Calculate boundaries
    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    # Identify outliers
    outliers_df = invent_df.filter(
        (col(column_name) < lower_bound) |
        (col(column_name) > upper_bound)
    )

    print("Column:", column_name)
    print("Q1:", q1)
    print("Q3:", q3)
    print("IQR:", iqr)
    print("Lower Bound:", lower_bound)
    print("Upper Bound:", upper_bound)
    print("Outlier Count:", outliers_df.count())

    display(outliers_df)

    return outliers_df

In [0]:
print("INVENTORY OUTLIERS")

inventory_columns = invent_df.columns 

for c in ["quantity_change", "unit_cost"]:
    if c in inventory_columns:
        find_outliers(invent_df, c)




INVENTORY OUTLIERS
Column: quantity_change
Q1: -4.0
Q3: 3.0
IQR: 7.0
Lower Bound: -14.5
Upper Bound: 13.5
Outlier Count: 0


movement_id,product_id,store_id,movement_ts,movement_type,quantity_change,reference_id,unit_cost,warehouse_id,recorded_ts


Column: unit_cost
Q1: 2978.0
Q3: 13024.0
IQR: 10046.0
Lower Bound: -12091.0
Upper Bound: 28093.0
Outlier Count: 0


movement_id,product_id,store_id,movement_ts,movement_type,quantity_change,reference_id,unit_cost,warehouse_id,recorded_ts


In [0]:
from pyspark.sql.functions import col
ret_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(RETURNS_PATH)
)
def find_outliers(ret_df, column_name):

    # Calculate Q1 and Q3
    q1, q3 = ret_df.approxQuantile(
        column_name, [0.25, 0.75], 0.01
    )

    # Calculate IQR
    iqr = q3 - q1

    # Calculate boundaries
    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    # Identify outliers
    outliers_df = ret_df.filter(
        (col(column_name) < lower_bound) |
        (col(column_name) > upper_bound)
    )

    print("Column:", column_name)
    print("Q1:", q1)
    print("Q3:", q3)
    print("IQR:", iqr)
    print("Lower Bound:", lower_bound)
    print("Upper Bound:", upper_bound)
    print("Outlier Count:", outliers_df.count())

    display(outliers_df)

    return outliers_df

In [0]:
numeric_columns = [
    "return_quantity","refund_amount",
]

for column_name in numeric_columns:
    find_outliers(ret_df, column_name)

Column: return_quantity
Q1: 3.0
Q3: 7.0
IQR: 4.0
Lower Bound: -3.0
Upper Bound: 13.0
Outlier Count: 0


return_id,order_line_id,return_ts,return_quantity,refund_amount,return_reason,inspection_status,store_id,product_id,approved_flag


Column: refund_amount
Q1: 2884.0
Q3: 8767.0
IQR: 5883.0
Lower Bound: -5940.5
Upper Bound: 17591.5
Outlier Count: 0


return_id,order_line_id,return_ts,return_quantity,refund_amount,return_reason,inspection_status,store_id,product_id,approved_flag


In [0]:
# DBTITLE 1,21. Check Return Reasons

display(
    returns_df.groupBy("return_reason")
    .count()
)


# COMMAND ----------

# DBTITLE 1,22. Check Return Inspection Status

display(
    returns_df.groupBy("inspection_status")
    .count()
)


# COMMAND ----------

# DBTITLE 1,23. Source Data Summary

print("SALES COUNT:", sales_df.count())
print("INVENTORY COUNT:", inventory_df.count())
print("RETURNS COUNT:", returns_df.count())

print("Source Data Profiling Completed")

return_reason,count
CUSTOMER_REQUEST,103
DAMAGED,100
WRONG_ITEM,113


inspection_status,count
INACTIVE,92
ACTIVE,115
PENDING,109


SALES COUNT: 11412
INVENTORY COUNT: 634
RETURNS COUNT: 316
Source Data Profiling Completed
